# DSA 405 P2 — House PTR PDF extraction and cleaning audit

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StrokeOfLuck/dsa405-part-2/blob/main/notebooks/DSA405_002_FA26_P2_sryan3.ipynb)

**What changes from P1:** Start with the archived **2025 House filing PDFs**. The original scraper's pinned Stage 3 turns those PDFs into a transaction CSV; Stage 4 resolves ticker and asset fields into a second CSV. This P2 notebook audits Stage 3 and logs the Stage 4 decisions. The published scraper repo is never modified.

**Unit:** one transaction row extracted from a PDF. `source_year=2025` is the filing index year, not necessarily the transaction year.

## 1. Make an isolated 2025 copy and rebuild the CSVs

The pinned source commit and checksums in `data/raw/2025_pdf_manifest.csv` identify 515 PDFs. The committed manifest stays unmodified in `data/raw/`. The script copies and verifies the PDFs in `data/work/01_pdfs/2025` and copies the 2025 XML index into the working directory. The large PDF copies are downloaded on first run; later runs reuse them. This step requires Git and network access the first time. It may take several minutes on CPU.

### Read the setup cell, line by line

| Line(s) | What the code does | Why it is here |
|---|---|---|
| 1 | Imports `Path`, which represents filesystem paths. | The notebook needs to find the project and name its output files. |
| 2 | Imports `os`, `subprocess`, and `sys`. | These change directories, run Git/Python commands, and select the current Python interpreter. |
| 4–5 | Checks whether `scripts/rebuild_2025.py` exists in the current directory. | Colab opens a single notebook; a local reader may already be in the project root. |
| 6–7 | If the script is in the parent directory, moves there. | Jupyter may start inside `notebooks/` in a local checkout. |
| 8–12 | Otherwise names a `dsa405-part-2` directory, clones this repo there only if the script is absent, then moves into it. | Colab needs the script, manifest, and requirements in addition to the notebook; the existence check avoids a redundant clone. `check=True` stops if cloning fails. |
| 14–18 | Attempts to import `google.colab`; ignores `ImportError` locally and uses `else` when in Colab. | Package installation is only automatic in Colab; a local environment may already have dependencies. |
| 19 | Installs the repo's `requirements.txt` with the same Python interpreter that runs this notebook. | Stage 3 and Stage 4 need their declared Python dependencies. |
| 21–22 | Names `data/work/rebuild.log` and creates its parent folder. | Keep verbose pipeline output in a reproducible working location. |
| 23–24 | Opens the log for writing and runs `scripts/rebuild_2025.py`, sending normal output and errors into that file. | The script copies and checks the 2025 PDFs, then creates the Stage 3 and Stage 4 CSVs; the notebook stays readable. |
| 25–26 | Prints the exit code and last 18 log lines. | Quickly see whether the pipeline succeeded and which stage just finished. |
| 27–28 | Raises an error when the exit code is nonzero. | Later audit cells must not silently use stale or incomplete CSVs. |

**What to look for:** `Pipeline result: 0` means the script finished. On the first Colab run it downloads the PDF copies; later runs can reuse verified copies. If it fails, open `data/work/rebuild.log` for the full error. Nothing in this cell writes to the original scraper repository.

In [ ]:
from pathlib import Path
import os, subprocess, sys

# Colab opens the notebook file without cloning the rest of the repository.
if not Path("scripts/rebuild_2025.py").exists():
    if Path("../scripts/rebuild_2025.py").exists():
        os.chdir("..")
    else:
        target = Path("dsa405-part-2")
        if not (target / "scripts/rebuild_2025.py").exists():
            subprocess.run(["git","clone","--depth","1","https://github.com/StrokeOfLuck/dsa405-part-2.git",str(target)],check=True)
        os.chdir(target)

try:
    import google.colab  # available in Colab, absent in a normal local notebook
except ImportError:
    pass
else:
    subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"],check=True)

log_path=Path("data/work/rebuild.log")
log_path.parent.mkdir(parents=True,exist_ok=True)
with log_path.open("w") as log:
    result=subprocess.run([sys.executable,"scripts/rebuild_2025.py"],stdout=log,stderr=subprocess.STDOUT)
print("Pipeline result:",result.returncode)
print("\n".join(log_path.read_text(errors="replace").splitlines()[-18:]))
if result.returncode:
    raise RuntimeError(f"Pipeline failed. Inspect {log_path} for the stage and error.")

The pinned original PDFs remain untouched in the scraper archive; the P2 working copies are checked against the committed manifest. Stage 3 writes `transactions_raw.csv`; Stage 4 writes `transactions_resolved.csv`. Publication makes a web-facing CSV too. No source PDF is changed.

## 2. Audit the Stage 3 transaction CSV

### Read the CSV loading cell, line by line

| Line(s) | What the code does | Why it is here |
|---|---|---|
| 1 | Imports pandas as `pd`. | We use DataFrames to compare the transaction CSVs. |
| 2 | Imports `display`. | Tables render legibly inside Jupyter and Colab. |
| 4 | Points `source` at the Stage 3 transaction CSV. | This is the first parser output to audit. |
| 5 | Points `resolved` at the Stage 4 CSV. | We compare the later ticker resolver against Stage 3. |
| 6–7 | Loads each CSV with every column as text; disables automatic blank-to-NaN conversion. | IDs and source text retain their exact form, and an empty string is consistently counted as missing. |
| 8 | Prints both `(row count, column count)` pairs. | A quick check that both files loaded and Stage 4 did not unexpectedly lose rows. |
| 9 | Counts copied `*.pdf` files in the 2025 work folder. | Compares local inputs against the 515-file manifest. |
| 10 | Counts each `source_year` value. | Confirms this run uses the 2025 filing index, which can include earlier trade dates. |
| 11 | Displays five rows with identifiers, parsed values, and a review flag. | Lets you inspect the unit of analysis and see how an extracted transaction looks. |

**What to look for:** the expected rebuild has 515 copied PDFs and 7,667 transaction rows in each CSV. Counts describe this pinned snapshot; investigate if a future input changes them.

In [ ]:
import pandas as pd
from IPython.display import display

source=Path("data/work/04_transactions/transactions_raw.csv")
resolved=Path("data/work/04_transactions/transactions_resolved.csv")
raw=pd.read_csv(source,dtype=str,keep_default_na=False)
clean=pd.read_csv(resolved,dtype=str,keep_default_na=False)
print("V8.1 extracted:",raw.shape,"V8.2 resolved:",clean.shape)
print("Source PDFs copied:",len(list(Path("data/work/01_pdfs/2025").glob("*.pdf"))))
print("Source year:",raw.source_year.value_counts(dropna=False).to_dict())
display(raw[["filing_id","transaction_number_in_filing","politician","asset","ticker","transaction_date","amount_category","needs_review"]].head(5))

### Column inventory

Read identifiers as strings to preserve their printed form. Empty strings count as missing. Audit every field used in this notebook before looking at the resolved output.

### Read the inventory cell, line by line

| Line | What the code does | Why it is here |
|---|---|---|
| 1 | Lists the 19 Stage 3 fields we will inspect, including raw source text, normalized values, provenance, and review flags. | Keeps the audit focused and ensures every later dictionary entry has a named field. |
| 2 | Assigns an intended meaning/type to each listed field. | CSV columns were loaded as strings to protect IDs; intended types still need documenting. |
| 3 | Asserts all 19 names actually occur in `raw`. | A changed parser schema should stop the notebook before producing misleading results. |
| 4 | For each field, calculates loaded dtype, intended type, empty-string count/rate, and distinct-value count, then builds `audit`. `round(...,4)` shows missing share to four decimals. | Gives a reproducible variable inventory instead of guessing from a few sample rows. |
| 5 | Displays the inventory table. | Check high missing rates and whether identifiers are being treated as text. |

**What to look for:** a missing value here is exactly `""` in the CSV; it does not by itself mean the underlying PDF lacked the information.

In [ ]:
FIELDS=["filing_id","transaction_number_in_filing","politician","source_year","owner","asset","ticker","asset_type","transaction_type","transaction_date","amount_min","amount_max","amount_status","needs_review","review_reason","original_pdf_url","asset_raw","transaction_date_raw","amount_raw"]
TYPES={"filing_id":"string ID","transaction_number_in_filing":"integer","politician":"string","source_year":"integer","owner":"category","asset":"string","ticker":"string","asset_type":"category","transaction_type":"category","transaction_date":"date","amount_min":"USD lower bound","amount_max":"USD upper bound","amount_status":"category","needs_review":"boolean","review_reason":"string","original_pdf_url":"URL","asset_raw":"source text","transaction_date_raw":"source text","amount_raw":"source text"}
assert set(FIELDS)<=set(raw.columns)
audit=pd.DataFrame([{"variable":x,"loaded dtype":str(raw[x].dtype),"intended type":TYPES[x],"missing count":int(raw[x].eq("").sum()),"missing rate":round(raw[x].eq("").mean(),4),"distinct including blank":int(raw[x].nunique(dropna=False))} for x in FIELDS])
display(audit)

### Read the distribution and range checks, line by line

| Line | What the code does | Why it is here |
|---|---|---|
| 1–3 | Loop over audited fields and print value counts for fields with fewer than 30 distinct entries. | Show all observed levels of small categorical fields without dumping large free-text columns. `dropna=False` would retain nulls if any existed. |
| 4 | Selects the row number and numeric amount-bound fields. | These should support meaningful numeric range checks. |
| 5 | Replaces blanks with missing values and parses numbers; invalid nonblank values become `NaN`. | A blank is distinct from a malformed numeric entry. The original text remains untouched. |
| 6 | Prints minimum, maximum, span, and count of nonblank values that failed parsing. | Look for impossible row numbers or dollar bounds and quantify malformed entries. |
| 7 | Defines the two date fields and their expected formats: ISO for normalized date, slashed US date for raw text. | Source-like text and parser output should not be tested with the same format. |
| 8 | Attempts strict parsing after excluding blanks; failures become missing parsed dates. | Detects raw text with extra PDF fragments as well as invalid calendar dates. |
| 9 | Prints the date range and nonblank unparsed count for each field. | Distinguishes a parser's normalized date from a messy extraction string. |

**What to look for:** `transaction_date_raw` can fail strict standalone-date parsing because neighboring PDF text was captured; inspect examples in the next cell before calling the normalized date incorrect.

In [ ]:
for field in FIELDS:
    if raw[field].nunique(dropna=False)<30:
        print(field,raw[field].value_counts(dropna=False).to_dict())
for field in ["transaction_number_in_filing","amount_min","amount_max"]:
    parsed=pd.to_numeric(raw[field].replace("",pd.NA),errors="coerce")
    print(field,"min",parsed.min(),"max",parsed.max(),"range",parsed.max()-parsed.min(),"nonempty invalid",int((parsed.isna()&raw[field].ne("")).sum()))
for field,fmt in [("transaction_date","%Y-%m-%d"),("transaction_date_raw","%m/%d/%Y")]:
    parsed=pd.to_datetime(raw[field].replace("",pd.NA),format=fmt,errors="coerce")
    print(field,"min",parsed.min(),"max",parsed.max(),"nonempty unparsed",int((parsed.isna()&raw[field].ne("")).sum()))

### Read the integrity checks, line by line

| Line(s) | What the code does | Why it is here |
|---|---|---|
| 1 | Defines a proposed composite key: filing ID plus transaction position in that filing. | One filing can contain many transaction rows. |
| 2–3 | Starts a named set of checks and counts exactly identical rows. | An exact duplicate could inflate totals. |
| 4 | Counts duplicate composite keys. | Two different records with the same proposed identifier need investigation. |
| 5 | Trims/lowercases audited fields and counts placeholder strings such as `n/a` and `999`. | These may be disguised missing values rather than real data. |
| 6 | Counts Unicode replacement characters across the fields. | `�` can indicate damaged text encoding. |
| 7 | Counts numeric looking filing IDs with a leading zero. | IDs must stay strings so that a leading zero would survive CSV loading. |
| 8 | Counts rows whose asset or politician field consists solely of `total` or `subtotal`. | A PDF table summary could accidentally be parsed as a trade. |
| 9 | Converts both amount bounds for comparison and counts rows where minimum exceeds maximum. | A reversed range is internally inconsistent. |
| 10–12 | Counts parser review flags, closes the dictionary, and displays the named results. | A flag marks a need to inspect the source PDF, not a proven error. |
| 13 | Strictly parses the raw date as a standalone `MM/DD/YYYY` string. | Tests the extraction text separately from the normalized date. |
| 14 | Marks nonempty raw-date strings that fail that strict parse. | Finds strings with attached text and invalid dates. |
| 15 | Counts those exceptions and the subset that Stage 3 marked `False` for `needs_review`. | Shows the gap between this class audit and existing parser flags. |
| 16 | Displays eight exceptions with identifiers, both date fields, review flag, and source PDF URL. | Gives concrete cases to inspect in the actual disclosure. |

**What to look for:** 640 raw-date strings fail strict standalone parsing in this snapshot; 617 were unflagged. This check identifies an extraction-text issue; it does not prove that the normalized transaction date is wrong.

In [ ]:
keys=["filing_id","transaction_number_in_filing"]
checks={
 "exact duplicates":int(raw.duplicated().sum()),
 "duplicate filing+row keys":int(raw.duplicated(keys).sum()),
 "sentinel tokens":int(sum(raw[x].str.strip().str.lower().isin(["n/a","na","null","none","999"]).sum() for x in FIELDS)),
 "replacement character (encoding)":int(sum(raw[x].str.contains("\ufffd",regex=False).sum() for x in FIELDS)),
 "leading-zero IDs":int(raw.filing_id.str.match(r"^0[0-9]+$").sum()),
 "total/subtotal labels":int(sum(raw[x].str.contains(r"^\s*(?:total|subtotal)\s*$",case=False,regex=True).sum() for x in ["asset","politician"])),
 "nonempty amount min > max":int((pd.to_numeric(raw.amount_min,errors="coerce")>pd.to_numeric(raw.amount_max,errors="coerce")).sum()),
 "review flagged":int(raw.needs_review.eq("True").sum()),
}
display(pd.Series(checks,name="count").to_frame())
raw_date=pd.to_datetime(raw.transaction_date_raw.replace("",pd.NA),format="%m/%d/%Y",errors="coerce")
bad_date=raw.transaction_date_raw.ne("")&raw_date.isna()
print("Non-standalone raw date strings:",int(bad_date.sum()),"of which scraper did not flag:",int((bad_date&raw.needs_review.eq("False")).sum()))
display(raw.loc[bad_date,["filing_id","transaction_number_in_filing","transaction_date_raw","transaction_date","needs_review","original_pdf_url"]].head(8))

**Check the exceptions:** Text beside a date may have bled in from an adjacent PDF cell. This does not automatically make the resolved ISO date wrong. Open the linked original PDF for examples before changing them. Explicitly record defect classes checked and not found above.

## 3. Data dictionary for Stage 3 fields

### Read the data dictionary cell, line by line

| Line | What the code does | Why it is here |
|---|---|---|
| 1–3 | Creates `NOTES`, one interpretation and caveat for each of the 19 audited fields. | For example, `source_year` is the filing-index year and an amount bound is not an exact trade size. |
| 4 | Sets expected ranges or values for fields where we have a defensible rule. | Avoids inventing ranges for source-defined free text. |
| 5 | Builds one row per field using its intended type, units, observed factor levels when there are fewer than 30, expected range, blank count, missing interpretation, and field note. | The dictionary documents the actual loaded extract along with its limitations. `RANGES.get` supplies a generic PDF-check reminder for unspecified fields. |
| 6 | Renders the complete dictionary. | Use it in the assignment to explain variables and the meaning of missingness. |

**What to look for:** observed category levels come from this CSV, while the notes and expected ranges are interpretations to verify against the disclosure and parser code.

In [ ]:
NOTES={
 "filing_id":"House filing ID; keep as text, not a person identifier", "transaction_number_in_filing":"Row position in one filing; use with filing_id as proposed key", "politician":"Reported member name; not a stable ID", "source_year":"Index year, not necessarily trade year", "owner":"May be blank if no separate owner given", "asset":"V8.1 parser text; V8.2 preserves as asset_v8_1", "ticker":"V8.1 parsed candidate; V8.2 preserves as ticker_v8_1", "asset_type":"Source disclosure asset class", "transaction_type":"P purchase, S sale, E exchange; partial sales retain suffix", "transaction_date":"Reported trade date, not filing date", "amount_min":"Minimum of disclosed band, not an exact amount", "amount_max":"Maximum of disclosed band; blanks may mean open ended or uncertain", "amount_status":"Parser's amount classification", "needs_review":"Parser-generated flag, not proof that unflagged rows are correct", "review_reason":"Parser reasons; may be blank when not flagged", "original_pdf_url":"Link to the official source PDF", "asset_raw":"Text from source extraction; retain for reversibility", "transaction_date_raw":"Source-like text can contain neighboring PDF fragments", "amount_raw":"Source-like amount text; preserve alongside numeric bounds"
}
RANGES={"source_year":"2025","transaction_number_in_filing":"positive integer","transaction_date":"valid calendar date","amount_min":"nonnegative USD","amount_max":"nonnegative USD or blank for nonstandard cases","needs_review":"True / False"}
dictionary=pd.DataFrame([{"variable":x,"type":TYPES[x],"units":"USD" if x in ["amount_min","amount_max"] else ("calendar date" if x=="transaction_date" else "code/text or none"),"factor levels":", ".join(sorted(raw[x].unique())) if raw[x].nunique(dropna=False)<30 else "—", "valid range":RANGES.get(x,"Source-defined; check PDF"),"missing count":int(raw[x].eq("").sum()),"missing means":"Blank in parser output; confirm against PDF" if raw[x].eq("").any() else "None in this extract", "notes":NOTES[x]} for x in FIELDS])
display(dictionary)

## 4. Quantified extraction and cleaning decisions

Stage 3 extracts values from PDF text and writes V8.1. Stage 4 resolves ticker and display-asset fields to V8.2. This log counts each decision **on the 2025 data** and names the original evidence that lets a reader reverse it. The source code for both stages is pinned in the manifest. The class-specific check adds a flag for raw date text that includes neighboring content, without changing the reported transaction date.

### Read the decision log and P2 cleaning cell, line by line

This cell records transformations **with a count, reason, information loss, and reversal path**. “Rows/cells affected” is a count for that decision; counts from different decisions can overlap and should not be added as a number of unique transactions.

| Line(s) | What the code does | Why it is here |
|---|---|---|
| 1 | Asserts Stage 3 and Stage 4 have equal row counts. | A value-level comparison requires the same number of records. |
| 2 | Compares the two key columns in row order after resetting indexes. | Avoids accidentally comparing unrelated rows after a reorder. |
| 3 | Rejects duplicate composite keys. | Each row must correspond to exactly one transaction for this comparison. |
| 4 | Creates an empty list of decisions. | The list will become the displayed audit log. |
| 5–6 | Defines `log_decision` and appends a numbered record with stage, field, action, affected count, reason, loss, and reversal. | Gives every decision the same auditable format. |
| 8–12 | Names three Stage 3 comparisons: raw versus parsed asset, transaction type, and date, each with its rationale. | The parser converts messy PDF text to analytic fields. |
| 13–14 | Counts unequal raw/parsed values and logs each transformation. | Quantifies transformations while preserving the original extraction text for review. |
| 15 | Logs nonblank Stage 3 ticker candidates. | A candidate was inferred from asset text; `asset_raw` and PDF link remain as evidence. |
| 16–18 | Counts each `amount_status` class and logs its explanation. | Amount-band parsing can be exact, nonstandard, or uncertain; the raw amount is retained. |
| 20–23 | Counts each Stage 4 ticker status and logs accepted, inapplicable, or ambiguous classifications. | A classification can happen even if its visible ticker value stays the same. |
| 24–27 | Compares six resolved fields with their retained V8.1 versions and counts changed values. | Separates actual replacements from the classification count. |
| 28 | Prints the six change counts. | Zero is informative and should be reported as zero. |
| 29–32 | For a field with changed values, finds its matching V8.1 column and logs a reversible replacement. | The old value is stored beside the new one. This loop contributes no entries when all counts are zero. |
| 33 | Logs six preserved V8.1 columns for every resolved row. | Documents the added before/after evidence; this is a cell count (`rows × 6`), not a row count. |
| 35–36 | Copies the Stage 4 DataFrame into `p2`. | P2 audit columns should not overwrite the loaded Stage 4 comparison table. |
| 37–39 | Checks whether each raw date is *exactly* a slashed date and flags text that is not. | Captures PDF text contamination without changing the normalized date. |
| 40 | Extracts an initial `MM/DD/YYYY` token, or an empty string if absent. | A leading date may still be usable when the raw field contains trailing text. |
| 41 | Parses that token and formats it as ISO `YYYY-MM-DD`; failures become empty. | Makes its representation comparable to `transaction_date`. |
| 42 | Flags cases where a parsed prefix exists but differs from the normalized transaction date. | A disagreement deserves source-PDF review. |
| 43–45 | Adds three P2 decisions and their observed counts to the log. | Makes the new audit fields explicit and reversible. |
| 46–47 | Names `data/clean/house_ptr_2025_p2.csv` and creates its folder. | P2 output lives separately from immutable raw inputs and working Stage 3/4 CSVs. |
| 48 | Writes the P2 DataFrame without a pandas index column. | Produces the actual CSV for the assignment. |
| 49–50 | Converts decisions to a DataFrame and displays them. | The audit log can be read alongside the created file. |
| 51–52 | Prints the output path and the two new date-flag counts. | Gives a quick final check of what the P2 audit added. |

**What to look for:** in the pinned batch the six direct Stage 4 comparisons are all zero. The date-prefix disagreement count is zero too, despite extra text in some raw date strings. The classification decisions still matter; inspect the original PDFs before making a manual correction.

In [ ]:
assert len(raw)==len(clean),"Stage 4 unexpectedly changed row count"
assert raw[keys].reset_index(drop=True).equals(clean[keys].reset_index(drop=True)),"Stage 4 changed transaction order or keys"
assert raw.duplicated(keys).sum()==0,"Resolve duplicate keys before accepting one-to-one comparison"
changes=[]
def log_decision(stage,column,action,count,reason,lost,reverse):
    changes.append({"#":len(changes)+1,"stage":stage,"column":column,"change made":action,"rows/cells affected":int(count),"why":reason,"what is lost":lost,"how to reverse":reverse})

# Stage 3: compare the preserved extraction text with its parsed field.
for original,parsed,reason in [
 ("asset_raw","asset","Separate the displayed asset from ticker and nearby PDF text for analysis"),
 ("transaction_type_raw","transaction_type","Isolate the P/S/E transaction code from adjacent PDF text"),
 ("transaction_date_raw","transaction_date","Convert source-like date text to an ISO date for chronological checks")]:
    count=int(raw[original].ne(raw[parsed]).sum())
    log_decision("3: PDF extraction",parsed,f"Parsed {original} into {parsed}",count,reason,f"Nothing from {original}; source-like text remains in the same row.",f"Re-read {original} and inspect original_pdf_url; re-run pinned Stage 3.")
log_decision("3: PDF extraction","ticker","Extracted ticker candidates from asset text",int(raw.ticker.ne("").sum()),"Source PDF asset text can contain a parenthetical stock symbol.","No raw text lost; asset_raw remains alongside ticker.","Compare ticker to asset_raw and the original PDF; re-run Stage 3.")
for status,count in raw.amount_status.value_counts(dropna=False).items():
    reasons={"valid_range":"Two source dollar values match a recognized disclosure band.","nonstandard_exact":"The source displays a nonstandard exact amount rather than a standard band.","missing_range_bound":"The source range cannot safely supply both numeric bounds."}
    log_decision("3: PDF extraction","amount_status",f"Classified amount as {status}",count,reasons.get(status,"Amount parser classified source text."),"Raw amount text remains in amount_raw; numeric bounds may be absent when uncertain.","Compare amount_raw, amount_min, amount_max and the source PDF; re-run Stage 3.")

# Stage 4: log classifications, and quantify actual changed values separately.
for status,count in clean.ticker_parse_status.value_counts(dropna=False).items():
    reasons={"accepted":"A source-structured ticker candidate was accepted.","not_applicable":"The row is not a stock asset, so stock ticker resolution does not apply.","ambiguous_preserved":"The candidate is too uncertain to assert as a ticker."}
    log_decision("4: ticker resolver","ticker_parse_status",f"Classified ticker as {status}",count,reasons.get(status,"Resolver classification; inspect pinned Stage 4 code."),"No source evidence lost; V8.1 ticker and asset are retained.","Use ticker_v8_1, asset_v8_1, and asset_raw or re-run pinned Stage 4.")
actual_changes={current:int(clean[current].ne(clean[prior]).sum()) for current,prior in [
 ("ticker_v8_2_cleaned","ticker_v8_1"),("asset_v8_2_cleaned","asset_v8_1"),
 ("review_reason","review_reason_v8_1"),("review_level","review_level_v8_1"),
 ("needs_review","needs_review_v8_1"),("geometry_quality_score","geometry_quality_score_v8_1")]}
print("Stage 4 values changed (zero is a legitimate finding):",actual_changes)
for field,count in actual_changes.items():
    if count:
        prior={"ticker_v8_2_cleaned":"ticker_v8_1","asset_v8_2_cleaned":"asset_v8_1","review_reason":"review_reason_v8_1","review_level":"review_level_v8_1","needs_review":"needs_review_v8_1","geometry_quality_score":"geometry_quality_score_v8_1"}[field]
        log_decision("4: ticker resolver",field,"Replaced V8.1 value",count,"Resolver accepted a candidate or recalculated a ticker-specific review flag.",f"Nothing; {prior} retains the previous value.",f"Restore directly from {prior}.")
log_decision("4: ticker resolver","six V8.1 audit columns","Preserved original fields next to resolved values",len(clean)*6,"Readers need a reversible before/after record.","Nothing; columns are added copies.","Drop the six *_v8_1 columns to reverse this addition.")

# P2-specific audit flags: distinguish source text contamination from a wrong date.
p2=clean.copy()
source_date=p2.transaction_date_raw
standalone=source_date.str.fullmatch(r"\d{1,2}/\d{1,2}/\d{4}")
p2["raw_date_has_extra_text"]=~standalone
p2["raw_date_prefix"]=source_date.str.extract(r"^(\d{1,2}/\d{1,2}/\d{4})",expand=False).fillna("")
parsed_prefix=pd.to_datetime(p2.raw_date_prefix.replace("",pd.NA),format="%m/%d/%Y",errors="coerce").dt.strftime("%Y-%m-%d").fillna("")
p2["date_prefix_disagrees"]=parsed_prefix.ne("")&parsed_prefix.ne(p2.transaction_date)
log_decision("P2: audit","raw_date_has_extra_text","Flagged non-standalone source date text",int(p2.raw_date_has_extra_text.sum()),"Adjacent PDF text may be attached to a date and needs source review.","Nothing; original transaction_date_raw remains unchanged.","Remove the flag; compare original text with the source PDF.")
log_decision("P2: audit","raw_date_prefix","Extracted leading date token for checking",int(p2.raw_date_prefix.ne("").sum()),"Check whether the V8.1 date agrees with the visible leading date despite extra text.","Nothing; source-like text remains in transaction_date_raw.","Drop this derived column; extract again from transaction_date_raw.")
log_decision("P2: audit","date_prefix_disagrees","Flagged parsed date disagreements",int(p2.date_prefix_disagrees.sum()),"Any disagreement between source-like date prefix and normalized date warrants source review.","Nothing; both original and normalized dates remain.","Remove the derived flag and recompute from the two date fields.")
output=Path("data/clean/house_ptr_2025_p2.csv")
output.parent.mkdir(parents=True,exist_ok=True)
p2.to_csv(output,index=False)
log_df=pd.DataFrame(changes)
display(log_df)
print("P2 clean output:",output)
print("Raw date extra-text flags:",int(p2.raw_date_has_extra_text.sum()),"date-prefix disagreements:",int(p2.date_prefix_disagrees.sum()))

The final P2 CSV also contains three audit fields: `raw_date_has_extra_text` (boolean; true when the raw text is not a standalone date), `raw_date_prefix` (text date in MM/DD/YYYY form extracted from the start, blank if absent), and `date_prefix_disagrees` (boolean; true if that prefix differs from the parsed transaction date). Each uses the raw text, so the source remains recoverable.

Stage 4 made zero changes to the six compared value fields in this 2025 batch; its ticker classifications and audit columns are still documented above. Stage 3 did make substantive parsing decisions, including the PDF text fields shown in the log. The P2 flags are review aids, not claims that the already parsed transaction date is wrong. Inspect source PDFs and add a numbered decision for any manual correction.

## 5. Row and column accounting

### Read the row and column accounting, line by line

| Line | What the code does | Why it is here |
|---|---|---|
| 1 | Finds Stage 3 column names absent from the final P2 table. | Includes columns Stage 4 renamed or replaced; inspect the names before calling any data lost. |
| 2 | Finds final P2 column names absent from Stage 3. | Includes renamed Stage 4 fields and the three P2 audit fields. |
| 3 | Builds an accounting table: starting/ending rows, recorded removal counts, and starting/ending columns. | Reconciles the outputs at both row and field level. The three row-removal categories are zero because this notebook never filters rows. |
| 4 | Asserts the final row count equals the original. | Catches accidental dropping or multiplication of transactions. |
| 5 | Asserts the old count minus absent names plus new names equals the final count. | Checks the column inventory arithmetic. |
| 6 | Displays the reconciliation. | Use its counts in the assignment's cleaning summary. |
| 7–8 | Prints absent and new column names. | A name may change while its values survive in a V8.1 or V8.2 field. |
| 9–11 | Prints the three CSV paths. | Shows precisely which file is raw Stage 3, resolved Stage 4, and final P2. |

**What to look for:** 7,667 rows remain at each step; the P2 CSV has 64 columns. The displayed “removed/renamed” and “added/renamed” numbers describe name differences, not evidence that raw PDF data was deleted.

In [ ]:
removed=sorted(set(raw.columns)-set(p2.columns))
added=sorted(set(p2.columns)-set(raw.columns))
accounting=pd.DataFrame([("Raw V8.1 rows",len(raw)),("Exact duplicate rows removed",0),("Near-duplicate rows removed",0),("Other rows removed",0),("P2 clean rows",len(p2)),("Raw columns",raw.shape[1]),("Columns removed/renamed",len(removed)),("Columns added/renamed",len(added)),("P2 clean columns",p2.shape[1])],columns=["item","count"])
assert len(raw)==len(p2)
assert raw.shape[1]-len(removed)+len(added)==p2.shape[1]
display(accounting)
print("Removed or renamed:",removed)
print("Added or renamed:",added)
print("Raw V8.1 CSV:",source)
print("V8.2 CSV:",resolved)
print("P2 clean CSV:",output)

## 6. Provenance brief (under 200 words)

This dataset comes from 515 periodic transaction report PDFs in the U.S. House Clerk's 2025 disclosure archive. Members submit the reports to disclose transactions, and the House publishes the PDFs. This project copies the PDFs from a pinned archive of my House PTR scraper without modifying the archive. The same parser extracts transaction rows into a first CSV; its next stage resolves some ticker and asset names into a second CSV. Each row retains a link to its source disclosure, and uncertain cases can be checked against that PDF.

The dataset describes **reported transactions in filings indexed under 2025**, even when a transaction happened in an earlier year. Disclosed amounts are often ranges, not exact trade values. The parser can misread a PDF or miss a transaction; a row without a review flag is not independently verified. It cannot establish unreported trades or prove that a member personally executed a trade.

**Before submission:** Review example PDFs and adjust this brief if audit findings change the limits.

## Submission check

- [ ] Inspect representative review cases and raw-date exceptions against their linked PDFs.
- [ ] Record any manual decisions with exact affected counts, reasons, losses and reversals.
- [ ] Run notebook from a restarted kernel, save executed outputs, and verify the counts.
- [ ] Include the course's self-scored rubric with notebook and repo link.